# Тематическое моделирование с использованием аддитивной регуляризации

На этом занятии мы познакомимся с библиотекой [BigARTM](https://github.com/bigartm/bigartm), которая является самым гибким инструментом для решения задач тематического моделирования.

Полная документация по библиотеке доступна [здесь](https://bigartm.readthedocs.io/).

Библиотека разработана командой Константина Вячеславовича Воронцова, профессора Российской академии наук, главного в стране специалиста по тематическому моделированию.

В мире существует несколько библиотек, позволяющих решать рассматриваемую задачу (на Западе особенно популярна [Gensim](https://radimrehurek.com/gensim/) - topic modeling for humans), однако бенчмарки показывают, что **BigARTM быстрее**!

Из выступления автора на конференции DataFest (2018.04.28).

![](https://i.imgur.com/Hq4NtZO.png)

Для ОС на базе Linux библиотека доступна для установки из PyPI. Для установки на Windows требуется выполнить несколько [дополнительных шагов](https://bigartm.readthedocs.io/en/stable/installation/windows.html).

In [ ]:
! pip install bigartm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 22.3 MB/s eta 0:00:00


## Загрузка данных

Для тематического моделирования будем использовать набор данных [20 newsgroups](https://scikit-learn.org/0.19/datasets/twenty_newsgroups.html), загрузить который можно с использованием sklearn.

In [ ]:
from sklearn.datasets import fetch_20newsgroups

In [ ]:
# загружаем набор данных (оставляем только тело каждой новости)
dataset = fetch_20newsgroups(remove=('headers', 'footers', 'qutes'))

In [ ]:
import pandas as pd

In [ ]:
df = pd.DataFrame({'text': dataset.data, 'target': dataset.target})
df

,text,target
0,I was wondering if anyone out there could enli...,7
1,A fair number of brave souls who upgraded thei...,4
2,"well folks, my mac plus finally gave up the gh...",4
3,Robert J.C. Kyanko (rob@rjck.UUCP) wrote:\n> a...,1
4,"From article <C5owCB.n3p@world.std.com>, by to...",14
...,...,...
11309,DN> From: nyeda@cnsvax.uwec.edu (David Nye)\nD...,13
11310,"I have a (very old) Mac 512k and a Mac Plus, b...",4
11311,I just installed a DX2-66 CPU in a clone mothe...,3
11312,In article <1qkgbuINNs9n@shelley.u.washington....,1


Далее нам необходимо выполнить предварительную обработку текстов.

In [ ]:
import re
from nltk import word_tokenize
from nltk.corpus import stopwords
import string
import nltk
from nltk.stem import WordNetLemmatizer

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
def preprocess(text: str) -> str:
    pattern = re.compile(r'[^a-z]+')
    text = text.lower()
    text = pattern.sub(' ', text).strip()

    word_list = word_tokenize(text)
    stopwords_list = set(stopwords.words('english'))

    word_list = [word for word in word_list if word not in stopwords_list]
    word_list = [word for word in word_list if len(word) > 2]
    word_list = [word for word in word_list if word not in string.punctuation]

    lemma = WordNetLemmatizer()
    word_list = [lemma.lemmatize(word) for word in word_list]
    text= ' '.join(word_list)

    return text

In [ ]:
df['text'] = df['text'].apply(lambda x: preprocess(x))

In [ ]:
df.head()

,text,target
0,wondering anyone could enlighten car saw day d...,7
1,fair number brave soul upgraded clock oscillat...,4
2,well folk mac plus finally gave ghost weekend ...,4
3,robert kyanko rob rjck uucp wrote abraxis iast...,1
4,article owcb world std com tombaker world std ...,14


Мы уже знаем, что в тематическом моделировании используется концепция "мешка слов". И у нас есть два основных формата сохранения данных для тематического моделирования:
- объект CountVectorizer из sklearn
- текстовый файл в формате VowpalWabbit

Данные загружаются в BigARTM в виде батчей (по умолчанию в батче 1000 документов).

### Кейс 1. CountVectorizer

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
vectorizer = CountVectorizer(ngram_range=(1,3), max_features=20000) # не забываем про n-грамы и фиксацию размера словаря
m = vectorizer.fit_transform(df['text'])

In [ ]:
n_wd = m.todense().T # получаем значения n_wd

In [ ]:
from artm import BatchVectorizer

In [ ]:
n_wd.shape

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


(20000, 11314)

In [ ]:
vocabulary = vectorizer.get_feature_names_out()

In [ ]:
vocabulary

array(['aa', 'aaa', 'aaron', ..., 'zubov', 'zuma', 'zvm'], dtype=object)

Загружаем и сохраняем батчи в папку

In [ ]:
batch_vectorizer = BatchVectorizer(data_format='bow_n_wd', n_wd=n_wd, vocabulary=vocabulary, target_folder='batches')

/usr/local/lib/python3.10/dist-packages/artm/batches_utils.py:227: DeprecationWarning: Please use `spmatrix` from the `scipy.sparse` namespace, the `scipy.sparse.base` namespace is deprecated.
  from scipy.sparse.base import spmatrix


### Кейс 2. Формат VowpalWabbit (если нужны модальности)

Ранее в теоретических выкладках мы работали только с понятием "терм" и в рамках решения задачи тематического моделирования искали распределение вероятностей $p(w|t)$.

В этом случае мы считаем, что все наши текстовые токены относятся к одной **модальности** - "слово".

В сложных задачах нам интересно не только представлять тему в виде распределения слов, но и в виде распределения именованных сущностей ($p(name|topic), p(date|topic)$) и др.

Пусть $W^m$ - множество терминов модальности $m$, $W^m\subseteq{W}$. Тогда мы можем записать функцию потерь в следующем виде:

$\sum_{m\in{M}}\tau_m\sum_{d\in{D}}\sum_{w\in{W^m}}n_{dw}ln(\sum_{t\in{T}}\phi_{wt}\theta_{td})+R(\Phi,\Theta)\rightarrow{max}$

E-шаг не изменится, а на M-шаге:

$\phi_{wt}=norm_{w\in{W^m}}(\sum_{d\in{D}}\tau_mn_{dw}p_{tdw}+\phi_{wt}\frac{\partial{R}}{\partial{\phi_{wt}}})$

$\theta_{td}=norm_{t\in{T}}(\sum_{w\in{d}}\tau_mn_{dw}p_{tdw}+\theta_{td}\frac{\partial{R}}{\partial{\theta_{td}}})$

**Замечание:** формально, биграммы - это новая модальность.

Формат VowpalWabbit имеет следующий вид:

doc_100500 |@words aaa:4 bbb:6 ccc ddd:2 |@names mikhail valera

Здесь мы выделили две модальности - обычные слова и имена.

**Замечание:** можно просто хранить свои документы в формате VowpalWabbit, игнорируя модальности (имея одну модальность можно не указывать id документа):

|text обязанность участие проект разработка внедрение программный обеспечение разрабатываемого компания автоматизированный система управление производство mes требование хороший понимание базовый вещь работать http websocker long polling etc умение верстать согласно макет typescript angular ngrx mobx nginx rxjs devextreme material angular jasmine опыт работа команда git тест баг трекинг опыт работа steam processing идеал kafka условия бело официально оклад премия компенсация мобильный связь интернет удалённый работа

In [ ]:
with open('vw.txt', 'w') as file:
  file.write('|text обязанность участие проект разработка внедрение программный обеспечение разрабатываемого компания автоматизированный система управление производство mes требование хороший понимание базовый вещь работать http websocker long polling etc умение верстать согласно макет typescript angular ngrx mobx nginx rxjs devextreme material angular jasmine опыт работа команда git тест баг трекинг опыт работа steam processing идеал kafka условия бело официально оклад премия компенсация мобильный связь интернет удалённый работа')

In [ ]:
batch_vectorizer2 = BatchVectorizer(data_format='vowpal_wabbit', data_path='vw.txt', target_folder='batches2')

После того, как батчи загружены, нам осталось получить словарь.

In [ ]:
import artm

Создание словаря по папке с батчами:

In [ ]:
dictionary = artm.Dictionary()
dictionary.gather(data_path='batches')

Создание словаря по объекту BatchVectorizer:

In [ ]:
dictionary2 = batch_vectorizer.dictionary

Словарь можно сохранить, чтобы в следующий раз не пересоздавать.

In [ ]:
dictionary.save(dictionary_path='dictionary')

In [ ]:
dictionary_load = dictionary.load(dictionary_path='dictionary.dict')

## LDA

Для начала давайте построим модель LDA. По умолчанию alpha и beta равны 0.01.

In [ ]:
lda = artm.LDA(num_topics=16, alpha=0.1, beta=0.1, num_document_passes=5, dictionary=batch_vectorizer.dictionary)

Обучение мы можем производить двумя способами:
- offline (классический) - проходим коллекцию несколько раз, обновляем $\Phi$ в конце каждого прохода;
- online (для очень больших коллекций) - проходим коллекцию один раз, обновляем $\Phi$ после прохода указанного количества батчей.

**Замечание:** матрица $\Theta$ обновляется на каждой итерации.

Мы будем использовать только offline-обучение.

In [ ]:
lda.fit_offline(batch_vectorizer, num_collection_passes=10)

In [ ]:
lda.phi_

,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,topic_10,topic_11,topic_12,topic_13,topic_14,topic_15
addition,7.088031e-05,0.000034,0.000226,2.297792e-04,7.009027e-05,0.000034,0.000070,1.005707e-04,0.000370,5.068561e-04,3.622363e-05,0.000150,0.000564,0.000121,2.782857e-04,0.000336
anyone,6.676551e-03,0.001482,0.004121,9.474669e-05,1.278330e-03,0.000129,0.000440,1.353650e-03,0.000171,6.216213e-04,1.301807e-03,0.000047,0.000729,0.002053,5.853242e-04,0.000606
anyone could,5.887441e-05,0.000006,0.000211,1.460299e-05,1.144981e-05,0.000002,0.000020,1.194290e-05,0.000003,5.333418e-06,1.914739e-06,0.000002,0.000003,0.000029,1.144071e-05,0.000003
body,1.184504e-05,0.000259,0.000278,3.322372e-05,1.283410e-04,0.000208,0.001480,1.943980e-04,0.000239,2.371192e-04,1.210646e-05,0.000010,0.000048,0.000494,2.390165e-03,0.000592
bumper,9.576983e-07,0.000002,0.000341,1.306034e-05,1.477084e-05,0.000001,0.000001,7.106984e-06,0.000002,8.552210e-07,1.016078e-06,0.000059,0.000001,0.000002,1.467465e-06,0.000003
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ordnance,7.540156e-07,0.000001,0.000001,7.259758e-07,6.828449e-07,0.000001,0.000001,1.005496e-06,0.000002,8.147642e-07,8.726951e-07,0.000562,0.000001,0.000001,7.767730e-07,0.000001
typical posting,7.540658e-07,0.000001,0.000001,8.252919e-07,1.233418e-04,0.000001,0.000001,9.976120e-07,0.000002,8.170449e-07,8.727700e-07,0.000002,0.000001,0.000001,8.090259e-07,0.000001
usr user,7.554296e-07,0.000001,0.000001,7.207876e-07,6.833019e-07,0.000001,0.000001,9.985474e-07,0.000002,8.592948e-07,9.069699e-07,0.000002,0.000001,0.000002,7.773872e-07,0.000160
embryo,7.540390e-07,0.000001,0.000001,1.118111e-06,6.828701e-07,0.000001,0.000001,1.001614e-06,0.000206,8.142723e-07,8.727604e-07,0.000002,0.000001,0.000001,9.645170e-06,0.000002


In [ ]:
lda.get_theta()

ValueError: ignored

По умолчанию матрица $\Theta$ не сохраняется. Предполагается, что мы будем тематизировать другие документы. Матрица $\Theta$ может быть вычислена по соответствующей формуле M-шага за одну итерацию (в виду того, что матрица $\Phi$ считается определенной).

Тематизируем коллекцию текстовых документов (предполагается, что она новая, но здесь в примере та же самая).

In [ ]:
lda.transform(batch_vectorizer)

,1000,1001,1002,1003,1004,1005,1006,1007,1008,1009,...,2990,2991,2992,2993,2994,2995,2996,2997,2998,2999
topic_0,0.945289,0.004285,0.000488,0.167806,0.000230,0.001754,0.000379,0.001230,0.966629,0.064532,...,0.038558,0.042885,0.002372,0.004942,0.001360,0.045605,0.009452,0.911072,0.000919,0.042625
topic_1,0.003429,0.004120,0.041369,0.009552,0.063628,0.091616,0.097155,0.025755,0.001313,0.096315,...,0.011674,0.007643,0.002041,0.009119,0.002808,0.038654,0.004042,0.007669,0.122115,0.081339
topic_2,0.006590,0.267147,0.001232,0.015439,0.000250,0.002617,0.001348,0.015583,0.000982,0.005344,...,0.009418,0.015852,0.001973,0.007422,0.003132,0.054535,0.003022,0.011482,0.001495,0.046597
topic_3,0.003310,0.005883,0.844925,0.005840,0.048317,0.061397,0.333703,0.039712,0.001027,0.108606,...,0.007203,0.011952,0.002517,0.102359,0.003686,0.032176,0.002344,0.009550,0.347205,0.044801
topic_4,0.003581,0.004058,0.012196,0.029965,0.000809,0.001443,0.311206,0.696877,0.001462,0.255956,...,0.117315,0.009979,0.001827,0.033004,0.004522,0.789905,0.003282,0.002262,0.001137,0.039335
topic_5,0.002866,0.002812,0.000294,0.003220,0.000185,0.019495,0.000423,0.000993,0.000693,0.003876,...,0.015734,0.004300,0.002740,0.006202,0.001496,0.001133,0.002298,0.001950,0.000569,0.046134
topic_6,0.003152,0.411416,0.000545,0.035099,0.000283,0.557455,0.000757,0.002230,0.000792,0.016939,...,0.006513,0.007064,0.003029,0.004856,0.004228,0.003648,0.002706,0.002489,0.000766,0.040506
topic_7,0.003443,0.005159,0.079890,0.209679,0.000309,0.245911,0.000581,0.001844,0.001091,0.005350,...,0.010292,0.044510,0.006874,0.014253,0.001984,0.003962,0.699956,0.006841,0.013187,0.031214
topic_8,0.003071,0.002690,0.000325,0.003120,0.000176,0.001270,0.000623,0.001573,0.001824,0.003439,...,0.303118,0.006171,0.102903,0.005001,0.001114,0.001379,0.002230,0.002332,0.000598,0.027958
topic_9,0.004633,0.004445,0.000343,0.474777,0.000191,0.001409,0.000448,0.001459,0.001346,0.005246,...,0.229614,0.790713,0.002765,0.005084,0.002730,0.001088,0.002708,0.003555,0.000743,0.034224


Смотрим ТОП-10 слов некоторых тем и пытаемся интерпретировать соответствующие темы.

In [ ]:
lda.phi_['topic_12'].nlargest(n=10)

key            0.020779
chip           0.009320
encryption     0.007975
bit            0.007685
use            0.007528
system         0.007033
data           0.006247
technology     0.005204
clipper        0.005091
information    0.004919
Name: topic_12, dtype: float32

In [ ]:
lda.phi_['topic_4'].nlargest(n=10)

god          0.017938
christian    0.007924
people       0.007869
would        0.007844
say          0.007316
one          0.007249
think        0.006999
believe      0.005295
know         0.004930
bible        0.004843
Name: topic_4, dtype: float32

Модель состоит из двух матриц, элементами которых являются соответственно p_wt ($\Phi$) и n_wt.

In [ ]:
lda.save('lda_p_wt', model_name='p_wt')
lda.save('lda_n_wt', model_name='n_wt')

Мы можем сохранить их отдельно, однако для тематизации новых документов (ввиду необходимости вычисления матрицы $\Theta$ мы должны загрузить обе матрицы).

In [ ]:
lda2 = artm.LDA(num_topics=20, alpha=0.1, beta=0.1, num_document_passes=5, dictionary=batch_vectorizer.dictionary)

In [ ]:
lda2.num_topics

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


20

In [ ]:
lda2.load('lda_p_wt', model_name='p_wt')
lda2.load('lda_n_wt', model_name='n_wt')

In [ ]:
lda2.num_topics

20

In [ ]:
lda.phi_

,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,topic_10,topic_11,topic_12,topic_13,topic_14,topic_15
addition,7.088031e-05,0.000034,0.000226,2.297792e-04,7.009027e-05,0.000034,0.000070,1.005707e-04,0.000370,5.068561e-04,3.622363e-05,0.000150,0.000564,0.000121,2.782857e-04,0.000336
anyone,6.676551e-03,0.001482,0.004121,9.474669e-05,1.278330e-03,0.000129,0.000440,1.353650e-03,0.000171,6.216213e-04,1.301807e-03,0.000047,0.000729,0.002053,5.853242e-04,0.000606
anyone could,5.887441e-05,0.000006,0.000211,1.460299e-05,1.144981e-05,0.000002,0.000020,1.194290e-05,0.000003,5.333418e-06,1.914739e-06,0.000002,0.000003,0.000029,1.144071e-05,0.000003
body,1.184504e-05,0.000259,0.000278,3.322372e-05,1.283410e-04,0.000208,0.001480,1.943980e-04,0.000239,2.371192e-04,1.210646e-05,0.000010,0.000048,0.000494,2.390165e-03,0.000592
bumper,9.576983e-07,0.000002,0.000341,1.306034e-05,1.477084e-05,0.000001,0.000001,7.106984e-06,0.000002,8.552210e-07,1.016078e-06,0.000059,0.000001,0.000002,1.467465e-06,0.000003
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ordnance,7.540156e-07,0.000001,0.000001,7.259758e-07,6.828449e-07,0.000001,0.000001,1.005496e-06,0.000002,8.147642e-07,8.726951e-07,0.000562,0.000001,0.000001,7.767730e-07,0.000001
typical posting,7.540658e-07,0.000001,0.000001,8.252919e-07,1.233418e-04,0.000001,0.000001,9.976120e-07,0.000002,8.170449e-07,8.727700e-07,0.000002,0.000001,0.000001,8.090259e-07,0.000001
usr user,7.554296e-07,0.000001,0.000001,7.207876e-07,6.833019e-07,0.000001,0.000001,9.985474e-07,0.000002,8.592948e-07,9.069699e-07,0.000002,0.000001,0.000002,7.773872e-07,0.000160
embryo,7.540390e-07,0.000001,0.000001,1.118111e-06,6.828701e-07,0.000001,0.000001,1.001614e-06,0.000206,8.142723e-07,8.727604e-07,0.000002,0.000001,0.000001,9.645170e-06,0.000002


In [ ]:
lda2.phi_['topic_15'].nlargest(n=10)

space          0.007742
one            0.005836
book           0.005301
list           0.005143
university     0.005065
new            0.004830
information    0.004825
well           0.004614
send           0.004239
request        0.003930
Name: topic_15, dtype: float32

Считаю, что здесь библиотека немного не доработана, и можно было бы автоматом изменять количество тем после загрузки тем. Будьте аккуратны при использовании :)

Давайте оценим качество полученной тематической модели с помощью перплексии.

In [ ]:
lda.perplexity_last_value

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


2676.467041015625

In [ ]:
lda.perplexity_value

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


[19625.2578125,
 5385.666015625,
 4869.08349609375,
 4131.5341796875,
 3507.171875,
 3153.1171875,
 2953.31494140625,
 2823.864501953125,
 2736.03857421875,
 2676.467041015625]

**Перплексия** (мера неопределенности слов в документе) является основной метрикой качества в тематическом моделировании (чем меньше значение, тем лучше).

$P(D)=exp(-\frac{1}{n}\sum_{d\in{D}}\sum_{w\in{d}}n_{dw}ln(p(w|d))), n = \sum_{d\in{D}}\sum_{w\in{d}}n_{dw}$

Построение тематических моделей обычно продолжают до сходимости перплексии.

## Аддитивная регуляризация

Теперь давайте построим модель общего вида.

In [ ]:
model = artm.ARTM(num_topics=16)

Можем добавлять интересующие нас метрики.

In [ ]:
model.scores.add(artm.PerplexityScore(name='perplexity_score')) # перплексия
model.scores.add(artm.TopTokensScore(name='top_tokens_score')) # топовые темы

In [ ]:
model.scores

[perplexity_score, top_tokens_score]

In [ ]:
model.num_document_passes = 3

In [ ]:
model.initialize(dictionary=dictionary)

In [ ]:
model.fit_offline(batch_vectorizer, num_collection_passes=5)

In [ ]:
model.score_tracker['perplexity_score'].last_value

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


4658.388671875

In [ ]:
model.score_tracker['perplexity_score'].value

[19743.92578125,
 5556.95947265625,
 5380.20458984375,
 5091.16064453125,
 4658.388671875]

Можем изменить параметры и выполнить еще несколько проходов.

In [ ]:
model.num_document_passes=10

In [ ]:
model.fit_offline(batch_vectorizer, num_collection_passes=3)

In [ ]:
model.score_tracker['perplexity_score'].value

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


[19743.92578125,
 5556.95947265625,
 5380.20458984375,
 5091.16064453125,
 4658.388671875,
 3791.626708984375,
 3201.696533203125,
 2910.474609375]

In [ ]:
model.score_tracker['top_tokens_score'].last_tokens

{'topic_0': ['window',
  'edu',
  'card',
  'problem',
  'one',
  'know',
  'thanks',
  'would',
  'anyone',
  'writes'],
 'topic_1': ['would',
  'one',
  'people',
  'right',
  'government',
  'like',
  'get',
  'state',
  'even',
  'really'],
 'topic_2': ['car',
  'writes',
  'com',
  'would',
  'article',
  'one',
  'edu',
  'like',
  'get',
  'anyone'],
 'topic_3': ['people',
  'year',
  'think',
  'president',
  'gun',
  'time',
  'right',
  'would',
  'country',
  'war'],
 'topic_4': ['god',
  'say',
  'would',
  'christian',
  'people',
  'like',
  'think',
  'know',
  'good',
  'even'],
 'topic_5': ['game',
  'team',
  'play',
  'new',
  'season',
  'year',
  'win',
  'player',
  'hockey',
  'league'],
 'topic_6': ['edu',
  'writes',
  'article',
  'apr',
  'one',
  'israel',
  'article apr',
  'know',
  'israeli',
  'writes article'],
 'topic_7': ['get',
  'would',
  'like',
  'know',
  'work',
  'year',
  'use',
  'make',
  'thing',
  'ground'],
 'topic_8': ['file',
  'window

В модели уже прослеживаются некоторые темы. Сразу скажу, что сохранение / загрузка матриц модели на диск осуществляется аналогично.

In [ ]:
model.transform(batch_vectorizer)

,1000,1001,1002,1003,1004,1005,1006,1007,1008,1009,...,2990,2991,2992,2993,2994,2995,2996,2997,2998,2999
topic_0,8.610771e-01,4.512261e-03,0.002001,3.528464e-01,0.000035,0.000476,0.000132,0.000145,9.771405e-01,8.769436e-02,...,5.680739e-02,3.463793e-02,1.086652e-03,1.555134e-06,0.000129,4.699222e-02,0.019793,8.683490e-01,0.000290,6.608335e-07
topic_1,5.956826e-08,4.724697e-04,0.108757,1.639246e-03,0.070679,0.093746,0.118406,0.051610,1.274887e-04,1.801490e-02,...,3.023293e-03,2.006946e-05,1.716466e-04,1.491087e-03,0.003192,5.825381e-02,0.000519,9.662313e-03,0.125133,1.224150e-03
topic_2,1.385728e-01,3.280173e-01,0.005523,7.380785e-03,0.000076,0.003617,0.004197,0.015492,9.959877e-04,4.147279e-05,...,3.955613e-03,5.244639e-03,9.419768e-04,1.382562e-05,0.002835,2.067046e-02,0.001267,1.163221e-01,0.008933,7.077146e-04
topic_3,2.981023e-08,2.012374e-03,0.706861,1.419630e-03,0.056112,0.061201,0.304508,0.017511,5.972080e-04,8.160628e-02,...,4.559466e-05,2.905662e-04,3.648106e-04,4.789078e-02,0.004862,3.566732e-02,0.000046,2.905323e-04,0.263614,3.153684e-06
topic_4,1.404876e-06,8.464188e-04,0.021243,6.659488e-03,0.038814,0.000166,0.224957,0.438459,1.052638e-03,2.272374e-01,...,8.610228e-02,1.814920e-04,1.414827e-05,1.083157e-02,0.009582,7.722625e-01,0.000293,3.221573e-07,0.002702,9.411309e-09
topic_5,9.070994e-10,1.717903e-05,0.000430,5.767004e-06,0.000041,0.016797,0.000252,0.000008,2.058623e-06,1.387763e-07,...,3.338646e-03,4.885879e-10,4.608971e-04,7.032439e-05,0.000186,2.691711e-05,0.000039,9.986429e-09,0.000006,9.632355e-05
topic_6,1.252174e-08,4.790269e-01,0.003928,2.561561e-04,0.001251,0.600801,0.003732,0.005434,1.196015e-05,1.880359e-02,...,1.128849e-04,4.263620e-06,1.589746e-03,1.850595e-05,0.029527,8.303540e-03,0.000608,3.918373e-07,0.000113,4.108323e-07
topic_7,3.046420e-07,4.687420e-02,0.097556,2.240352e-01,0.003417,0.213838,0.002480,0.001153,1.973007e-04,5.489430e-05,...,3.663920e-02,4.756688e-02,2.482974e-03,7.609014e-03,0.000986,1.369084e-02,0.605426,8.417100e-04,0.016541,4.456421e-06
topic_8,1.858583e-07,4.363678e-05,0.000279,1.133602e-05,0.000002,0.000042,0.000248,0.000474,2.217677e-03,6.786705e-09,...,3.673868e-01,6.488972e-02,1.539793e-01,2.446460e-06,0.000014,1.299418e-04,0.000029,2.124453e-06,0.000001,0.000000e+00
topic_9,3.421133e-04,1.046882e-03,0.000195,3.529662e-01,0.000039,0.000062,0.000571,0.000109,8.217314e-03,2.468019e-05,...,1.872726e-02,5.393678e-01,7.923311e-03,1.808985e-07,0.000042,8.925146e-05,0.015810,1.294841e-03,0.000073,1.105782e-08


Наконец, давайте добавим в модель пару регуляризаторов. Добавим регуляризатор для разреживания матрицы $\Phi$.

In [ ]:
model.regularizers.add(artm.SmoothSparsePhiRegularizer(name='smooth_sparse_phi_regularizer', tau=-4))

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Сбрасываем модель

In [ ]:
model.initialize(dictionary)

In [ ]:
model.master.clear_score_array_cache()

In [ ]:
model.fit_offline(batch_vectorizer, num_collection_passes=5)

In [ ]:
model.score_tracker['perplexity_score'].value

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


[19349.08984375,
 2500.562744140625,
 1863.4862060546875,
 1710.5496826171875,
 1628.1715087890625]

Видим, что модель сходится быстрее и перплексия превосходит LDA. Посмотрим на темы.

In [ ]:
model.phi_['topic_15'].nlargest(n=10)

one          0.010321
program      0.010085
sun          0.009439
request      0.009053
available    0.007972
anonymous    0.007820
entry        0.007705
name         0.007632
server       0.007589
send         0.006857
Name: topic_15, dtype: float32

In [ ]:
model.phi_['topic_5'].nlargest(n=10)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


new        0.015115
card       0.014356
color      0.013236
game       0.009572
video      0.009214
play       0.008736
monitor    0.008042
apple      0.007984
period     0.007979
driver     0.007461
Name: topic_5, dtype: float32

Регуляризаторы SmoothSparsePhiRegularizer и SmoothSparseThetaRegularizer осуществляют сглаживание или разрешение тем. Можем указать, какие конкретно темы мы хотим сглаживать или разреживать.

Сглаживание и разреживания приближают или отдаляют соответственно искомые распределения от константных. Чем больше мы разрежаем матрицу $\Phi$, тем дальше распределение $\phi_{wd}$ от константного. Аналогично с матрицей $\Theta$. Сглаживание работает наоборот.

$R(\Phi,\Theta)=\beta_0\sum_{t\in{T}}\sum_{w\in{W}}\beta_wln\phi_{wt}$, $\beta_0$ - коэффициент регуляризации.

Формула выше представлена в общем виде. Напоминаю, что в BigARTM искомые распределения сравниваются с равномерным константным распределением (все элементы выборки одинаковы).

Аналогично работает сглаживание или разреживание матрицы $\Theta$. При положительных коэффициентах регуляризации распределения стремятся быть похожими, а при отрицательных - далекими друг от друга.

Добавим другой регуляризатор.

In [ ]:
model.regularizers.add(artm.DecorrelatorPhiRegularizer(tau=5e5)) # имя можно не указывать - сформируется автоматически

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
model.regularizers

[smooth_sparse_phi_regularizer, 2:urn:uuid:022e2f1a-824b-11ee-9b27-0242ac1c000c]

In [ ]:
model.initialize(dictionary)
model.master.clear_score_array_cache()

In [ ]:
model.fit_offline(batch_vectorizer, num_collection_passes=5)

In [ ]:
model.score_tracker['perplexity_score'].value

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


[19349.08984375,
 2490.41259765625,
 1876.510498046875,
 1679.858154296875,
 1605.057861328125]

Регуляризатор DecorrelatorPhiRegularizer позволяет усилить различие между темами. Его формула:

$R(\Phi)=\tau{\sum_{t\in{T}}\sum_{s\in{T\setminus{t}}}\sum_{w\in{W}}\phi_{wt}{\phi_{ws}}}^T$

### Перплексия тестовой выборки

Когда мы реализуем итерационных процесс по EM-алгоритму, мы приближаемся к оптимальному решению, перплексия уменьшается. При этом, мы получаем оптимальную матрицу $\Theta$, которая является частью модели и заточена под ту коллекцию документов, на которой модели обучалась. Таким образом мы считаем перплексию обучающей выборки.

Для вычисления перплексии нам нужна матрица $\Theta$ для тестирующей выборки (она вычисляется в ходе выполнения метода transform).

In [ ]:
model.transform(batch_vectorizer)

,1000,1001,1002,1003,1004,1005,1006,1007,1008,1009,...,2990,2991,2992,2993,2994,2995,2996,2997,2998,2999
topic_0,6.125159e-01,2.927095e-01,6.512480e-02,2.893315e-01,5.767066e-02,5.918434e-02,1.060273e-02,4.675576e-02,2.675276e-01,1.899622e-01,...,1.259200e-01,7.598568e-03,2.076494e-01,1.637819e-07,7.903894e-03,9.765369e-02,9.475645e-02,4.337841e-01,4.526050e-03,3.787235e-09
topic_1,1.093951e-07,8.934189e-04,1.668396e-01,5.403557e-04,1.966584e-01,1.921223e-01,1.185379e-01,1.362126e-01,3.284164e-04,1.679474e-01,...,2.035598e-02,4.761236e-05,6.919795e-02,1.791114e-01,1.004382e-01,1.216066e-01,7.808520e-04,1.759146e-03,1.105728e-01,7.865790e-10
topic_2,1.101979e-01,2.422301e-01,1.743170e-02,8.806445e-02,1.506001e-04,1.356502e-01,1.783486e-02,1.980243e-02,1.166569e-01,1.815396e-01,...,1.470522e-02,2.153435e-03,5.371537e-02,8.848570e-06,5.968904e-02,2.140881e-03,4.473957e-02,3.697052e-01,1.838249e-01,1.061986e-03
topic_3,9.858619e-09,1.840360e-01,3.495726e-01,4.942337e-02,2.656821e-01,1.510103e-01,2.144339e-01,2.045359e-02,7.698978e-02,1.372449e-01,...,8.526219e-04,3.899586e-05,2.661483e-02,2.651865e-02,6.155343e-02,1.905569e-01,5.907582e-02,4.045914e-06,2.704575e-01,8.032590e-12
topic_4,8.872995e-04,1.564851e-03,8.209191e-03,9.387168e-02,5.660728e-03,4.579984e-05,1.334717e-01,2.043790e-01,3.273660e-02,5.807668e-02,...,1.451263e-01,7.459232e-05,1.032323e-03,2.436643e-02,3.364616e-04,3.123874e-01,1.663825e-04,2.259889e-02,2.875775e-02,0.000000e+00
topic_5,1.099275e-01,1.782509e-09,3.184455e-03,2.381924e-04,1.214920e-07,4.957084e-05,1.301064e-06,4.349245e-08,3.025704e-01,6.843067e-06,...,8.910326e-06,3.003997e-08,1.424354e-02,3.787514e-08,2.245498e-04,1.424999e-05,7.460734e-07,3.914119e-02,3.999912e-07,2.907566e-02
topic_6,9.809284e-09,1.690322e-01,5.585328e-02,4.820488e-07,7.964030e-03,1.387906e-01,2.028354e-02,5.896595e-02,1.102345e-02,1.510570e-02,...,8.205479e-04,1.707532e-07,1.058342e-04,4.204238e-03,1.244557e-01,1.514974e-02,5.475919e-06,1.304751e-12,8.248576e-05,9.030158e-08
topic_7,1.785934e-05,3.412504e-03,9.601203e-02,4.616816e-02,2.357650e-02,9.049425e-02,2.109287e-02,6.998441e-04,8.595884e-04,1.672422e-05,...,1.525719e-02,4.464292e-01,4.208785e-03,3.086886e-02,2.923145e-02,2.543290e-03,2.281743e-01,4.690157e-03,4.030143e-02,1.111802e-05
topic_8,8.725738e-06,2.360960e-05,1.770678e-07,1.978857e-09,1.778868e-09,1.403608e-09,2.752376e-08,3.144511e-05,3.054043e-02,1.808679e-13,...,3.480261e-01,1.118251e-02,1.099986e-03,5.851642e-03,1.397663e-04,3.424343e-02,6.197941e-08,1.309517e-11,6.086286e-10,0.000000e+00
topic_9,1.662129e-01,2.751811e-02,1.218283e-03,3.384996e-03,5.327139e-02,7.323365e-04,1.441447e-02,8.199999e-06,3.713370e-02,5.961314e-04,...,6.701253e-03,2.021408e-01,8.177955e-02,5.588360e-09,7.645061e-05,1.413580e-02,2.565774e-01,9.228032e-02,9.656229e-03,0.000000e+00


Вычисление перплексии для тестовой выборки выполняется по формуле:

$P(D)=exp(-\frac{1}{n''}\sum_{d\in{D'}}\sum_{w\in{d''}}n_{dw}ln(p(w|d))), n'' = \sum_{d\in{D}}\sum_{w\in{d''}}n_{dw}$

$d=d'\cup{d''}$ - случайное разбиение текстового документа d на 2 равные части; матрица $\Theta$ вычисляется по $d'$.

In [ ]:
model.get_score('perplexity_score').value

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


1565.64599609375

Вычисление перплексии для тестирующей выборки и своевременная остановка процесса обучения позволяет избежать переобучения тематической модели.

Информация о других регуляризаторах представлена [здесь](https://bigartm.readthedocs.io/en/stable/api_references/python_interface/regularizers.html).

Все основные метрики и регуляризаторы адаптированы для работы с мультимодальными данными. Более подробно узнать о мультимодальности в BigARTM можно [здесь](https://bigartm.readthedocs.io/en/stable/tutorials/python_userguide/m_artm.html).